# 8.3 • Does remembering a sequence improve the next prediction?

**Objectives:** explain LSTM memory and gates; construct input windows and next-step targets; prevent temporal leakage; compare a small PyTorch LSTM with persistence and lagged regression.

**Prerequisites:** lesson 8.1 and Python arrays. **Time:** 90–120 minutes. **Data:** an invented seasonal signal with disturbances and noise, not observations of vegetation, flow or climate. Values are in arbitrary units.

**Setup:** this lesson requires PyTorch in addition to the base course packages. From the repository root, install the base requirements, then for a Windows/Linux CPU environment run:

```bash
python -m pip install -r requirements.txt
python -m pip install "torch>=2.2,<3" --index-url https://download.pytorch.org/whl/cpu
```

For another platform use the [official installation selector](https://pytorch.org/get-started/locally/). `requirements-lstm.txt` is an alternative using your configured package index. Restart the kernel after installation. No GPU, credentials, pretrained weights or data downloads are used. Training runs by default; a missing PyTorch installation produces a clear setup error rather than silently skipping the model.

## What does an LSTM remember?

A long short-term memory network (LSTM) processes observations in order. At each timestep it updates a **cell state** (memory) and a **hidden state** (the output representation). Learned gates control that update:

- **Forget gate f:** how much old cell state to retain.
- **Input gate i:** how much candidate information g to add.
- **Output gate o:** how much of the updated memory to expose.

`new_cell = f * old_cell + i * g` and `new_hidden = o * tanh(new_cell)`.

Sigmoid keeps gate values between 0 and 1; tanh bounds candidate values. Gates are functions of the current input and previous hidden state, learned through training—not switches we manually set in the actual model. See [PyTorch LSTM documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html).

Predict: with the input gate closed, how does the forget gate affect how quickly an old memory fades?

In [ ]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from IPython.display import display
try:
    import torch
    from torch import nn
except ImportError as exc:
    raise ImportError('This lesson requires PyTorch. Follow the setup cell, then restart the kernel.') from exc

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course.py').exists())
OUT = ROOT / 'data' / 'generated' / 'lstm'
OUT.mkdir(parents=True, exist_ok=True)
torch.manual_seed(7)
torch.set_num_threads(1)  # Small CPU demonstration; avoid excessive thread overhead.
print('PyTorch:', torch.__version__, '| device: CPU')

# Illustrative gates only; the trained model learns its gates from inputs and state.
old_cell, forget, input_gate, candidate, output_gate = 0.6, 0.9, 0.8, 0.5, 0.7
new_cell = forget * old_cell + input_gate * candidate
new_hidden = output_gate * np.tanh(new_cell)
print(f'One memory update: cell={new_cell:.3f}; hidden={new_hidden:.3f}')
assert np.isclose(new_cell, 0.94)
fig, ax = plt.subplots(figsize=(6, 3))
steps = np.arange(16)
for gate in [0.2, 0.9, 1.0]:
    ax.plot(steps, gate**steps, marker='.', label=f'forget gate={gate}')
ax.set(xlabel='Timesteps', ylabel='Cell state', title='Memory retention with no new input (initial cell=1)')
ax.legend(); fig.tight_layout(); plt.show()

## Generate a timeline, then split it

The fixture combines seasonality, a slower oscillation, three invented disturbances and random noise. No model sees future noise or an advance warning of disturbances. A good forecast should still miss genuinely unexpected events.

Split the **raw timeline** first: days 0–599 train, 600–799 validate, 800–999 test. Generate windows separately inside each segment. This conservative choice discards the first 24 targets of each segment, preventing raw observations from appearing in multiple partitions.

Other forecasting designs may legitimately use pre-boundary history for the first test forecast, provided it was available at forecast time. Our strict disjoint-segment approach is easier to audit. Windows overlap *within* partitions, so their errors are dependent; do not treat them as independent samples for confidence intervals.

In [ ]:
rng = np.random.default_rng(42)
time = np.arange(1000)
signal = 0.7 * np.sin(2 * np.pi * time / 60) + 0.2 * np.sin(2 * np.pi * time / 170)
disturbance = np.zeros_like(signal)
for start, amplitude in [(210, -0.8), (640, 0.6), (850, -0.7)]:
    age = time - start
    active = age >= 0
    disturbance[active] += amplitude * np.exp(-age[active] / 12)
signal = signal + disturbance + rng.normal(0, 0.08, len(time))
LOOKBACK = 24
SEGMENTS = {'train': (0, 600), 'validation': (600, 800), 'test': (800, 1000)}
scaler = StandardScaler().fit(signal[:600, None])
scaled = scaler.transform(signal[:, None]).ravel()

def make_windows(values, start, stop, lookback):
    if lookback < 1 or stop - start <= lookback:
        raise ValueError('Need a positive lookback shorter than the segment.')
    target_times = np.arange(start + lookback, stop)
    input_times = target_times[:, None] - np.arange(lookback, 0, -1)
    assert input_times.min() >= start and input_times.max() < stop
    assert np.all(input_times < target_times[:, None])
    return (values[input_times, None].astype(np.float32),
            values[target_times, None].astype(np.float32), target_times, input_times)

data = {name: make_windows(scaled, start, stop, LOOKBACK) for name, (start, stop) in SEGMENTS.items()}
used = {name: set(np.concatenate([entry[2], entry[3].ravel()])) for name, entry in data.items()}
assert used['train'].isdisjoint(used['validation'])
assert used['train'].isdisjoint(used['test']) and used['validation'].isdisjoint(used['test'])
assert np.isclose(scaler.mean_[0], signal[:600].mean())
display(pd.DataFrame([{'split': name, 'windows': len(entry[0]), 'first_target_day': entry[2][0],
                       'last_target_day': entry[2][-1]} for name, entry in data.items()]))
fig, ax = plt.subplots(figsize=(11, 3))
for name, (start, stop) in SEGMENTS.items():
    ax.plot(time[start:stop], signal[start:stop], label=name)
ax.set(xlabel='Synthetic day', ylabel='Arbitrary units', title='One synthetic signal, chronological partitions')
ax.legend(); fig.tight_layout(); plt.show()

## Understand the tensor before the network

With `batch_first=True`, PyTorch expects **(batch, timesteps, features)**. Here one sample has shape `(24, 1)`: the last 24 observations of one signal. Its target is the next single observation, not the entire following season.

Our model uses one LSTM layer with eight hidden units followed by a linear output. Hidden and cell states reset to zeros for each window; this is a stateless window model. It has no memory outside those 24 inputs. Training uses backpropagation through time, which PyTorch computes automatically.

In [ ]:
class SmallLSTM(nn.Module):
    def __init__(self, hidden_size=8):
        super().__init__()
        self.memory = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, sequence):
        outputs, (hidden, cell) = self.memory(sequence)
        return self.output(outputs[:, -1, :])  # Last timestep, not last sample.

X_train, y_train = [torch.from_numpy(a) for a in data['train'][:2]]
X_val, y_val = [torch.from_numpy(a) for a in data['validation'][:2]]
network = SmallLSTM()
with torch.no_grad():
    output = network(X_train[:3])
assert X_train.shape[1:] == (LOOKBACK, 1) and output.shape == (3, 1)
assert sum(p.numel() for p in network.parameters()) == 361
print('Input:', tuple(X_train.shape), '| example output:', tuple(output.shape), '| parameters: 361')

## Train on the past, select with validation

Use mean squared error and Adam for 160 epochs. At each epoch, a full training batch updates weights; validation measures error without gradients. Save the lowest-validation-loss checkpoint. The epoch budget is fixed, and the final test period never selects the checkpoint.

Gradient clipping limits very large updates in recurrent training; it does not repair leakage or guarantee good predictions. The example intentionally keeps training small. Different versions/hardware can yield slightly different results even with fixed seeds.

In [ ]:
def fit_lstm(X_train, y_train, X_val, y_val, hidden_size=8, epochs=160, seed=7):
    torch.manual_seed(seed)
    model = SmallLSTM(hidden_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    history, best_loss, best_state, best_epoch = [], np.inf, None, None
    for epoch in range(epochs + 1):
        model.eval()
        with torch.no_grad():
            training_loss = criterion(model(X_train), y_train).item()
            validation_loss = criterion(model(X_val), y_val).item()
        assert np.isfinite([training_loss, validation_loss]).all()
        history.append({'epoch': epoch, 'training_mse': training_loss, 'validation_mse': validation_loss})
        if validation_loss < best_loss:
            best_loss, best_epoch = validation_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
        if epoch == epochs: break
        model.train()
        optimizer.zero_grad()
        training_error = criterion(model(X_train), y_train)
        training_error.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(history), best_epoch

model, history, best_epoch = fit_lstm(X_train, y_train, X_val, y_val)
assert history.training_mse.iloc[-1] < history.training_mse.iloc[0]
assert any(not torch.equal(model.state_dict()[name], network.state_dict()[name]) for name in model.state_dict())
print('Selected epoch:', best_epoch)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history.epoch, history.training_mse, label='training')
ax.plot(history.epoch, history.validation_mse, label='validation')
ax.axvline(best_epoch, color='gray', linestyle='--', label='selected checkpoint')
ax.set(xlabel='Epoch', ylabel='MSE in standardized units', title='LSTM learning curves')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'learning_curves.png', dpi=140); plt.show()
history.to_csv(OUT / 'loss_history.csv', index=False)

## Require the LSTM to earn its complexity

**Persistence** predicts the next value equals the last observed value. **Lagged ridge regression** predicts the next value using all 24 observed values, with a linear relationship and fixed regularization strength. Both use exactly the same target days as the LSTM. Ridge's alpha is fixed at 1 in advance; tune alternatives using validation only.

This is **rolling one-step-ahead evaluation with a fixed model**: at each test day, all prior observations up to yesterday are assumed available. Later windows include earlier *observed* test values, not earlier predictions. This is appropriate for that forecast protocol, but is not a multi-step forecast issued once on day 800.

Compute MAE and RMSE in original arbitrary units. The LSTM is not required to win. Differences on one invented signal do not establish general superiority.

In [ ]:
ridge = Ridge(alpha=1.0).fit(data['train'][0][:, :, 0], data['train'][1].ravel())
test_windows, _, target_days, _ = data['test']
with torch.no_grad():
    lstm_scaled = model(torch.from_numpy(test_windows)).numpy().ravel()
predictions = {
    'Persistence': test_windows[:, -1, 0],
    'Lagged ridge': ridge.predict(test_windows[:, :, 0]),
    'LSTM': lstm_scaled,
}
actual = signal[target_days]
rows, result = [], pd.DataFrame({'day': target_days, 'observed': actual})
for name, scaled_prediction in predictions.items():
    prediction = scaler.inverse_transform(scaled_prediction[:, None]).ravel()
    assert prediction.shape == actual.shape and np.isfinite(prediction).all()
    result[name] = prediction
    rows.append({'model': name, 'MAE': mean_absolute_error(actual, prediction),
                 'RMSE': np.sqrt(mean_squared_error(actual, prediction))})
scores = pd.DataFrame(rows)
display(scores)
assert np.allclose(result['Persistence'], signal[target_days - 1], atol=1e-6)
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
axes[0].plot(target_days, actual, color='black', linewidth=2, label='observed')
for name in predictions:
    axes[0].plot(target_days, result[name], alpha=0.8, label=name)
    axes[1].plot(target_days, result[name] - actual, alpha=0.8, label=name)
axes[0].set(ylabel='Arbitrary units', title='Held-out period: rolling one-step predictions, observed history')
axes[0].legend(ncol=4); axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(xlabel='Synthetic day', ylabel='Prediction − observation')
fig.savefig(OUT / 'forecast_comparison.png', dpi=140); plt.show()
scores.to_csv(OUT / 'test_metrics.csv', index=False)
result.to_csv(OUT / 'test_predictions.csv', index=False)

## When is persistence hard to beat?

For a random walk with independent zero-mean increments, the last value is the conditional-mean next-step forecast. A longer history cannot reveal the next random increment. The following separate NumPy demonstration illustrates the data-generating argument; it is not a second benchmark of our seasonal LSTM.

In [ ]:
walk_rng = np.random.default_rng(81)
walk = np.cumsum(walk_rng.normal(0, 0.1, 400))
walk_error = walk[1:] - walk[:-1]
print('Random-walk persistence RMSE:', np.sqrt(np.mean(walk_error**2)))
print('Known increment standard deviation: 0.1; more memory cannot predict independent future noise.')

## Your independent challenge

1. Predict what changes if the hidden size is 4 rather than 8. Fit with the same train/validation windows, seed and epoch budget. Compare minimum validation MSE and parameter counts; do not select using test scores.
2. Calculate a memory update with forget gate 0.2 and input gate 0.8. Explain why cell state and hidden state differ.
3. Propose a 12-versus-24 timestep comparison. To compare fairly, evaluate both on the **same target days** (start at day 624 for validation), and use the same training-only scaling statistics.
4. Explain how multi-step forecasting differs from the rolling protocol here. What happens if the next observed value is unavailable?

**Submit:** gate calculation, one controlled validation experiment, a forecast interpretation and a leakage checklist. Once test outcomes have been viewed, use newly reserved data for future final-performance claims.

In [ ]:
# Your validation-only experiment here; e.g., fit_lstm(..., hidden_size=4).

<details><summary>Solution guidance — open after trying</summary>

With the illustrative old cell 0.6, candidate 0.5 and input gate 0.8, the new cell is `0.2*0.6 + 0.8*0.5 = 0.52`; hidden state remains `0.7*tanh(0.52)`. A smaller hidden size reduces parameters, but the effect on validation loss is empirical. When changing lookback, intersect target-day arrays before comparing predictions; otherwise different days can explain the metric difference. For recursive multi-step forecasting, feed previous predictions back as inputs and evaluate error by lead time. Do not silently substitute future observations. In a real study reserve multiple time blocks, and locations too if claiming transfer to unseen sites.

</details>

## Check yourself and assessment

What do the three axes of the input tensor represent? Why split before constructing windows? Why is a rolling one-step evaluation not a season-ahead forecast? Can an LSTM learn an independent future noise realization?

| Criterion | Points | Evidence |
|---|---:|---|
| Memory concepts | 2 | Correct gate calculation and cell/hidden explanation |
| Temporal integrity | 3 | Past-only windows, raw-time separation, training-only scaling |
| Controlled experiment | 3 | Validation-only comparison on the same target days |
| Evaluation and communication | 2 | Baseline comparison, forecast protocol and limits |

**Mastery target:** 8/10, with no future information used in an input window.

## Transfer to environmental time series

Use documented observations with timestamps, units, missing-data flags and sampling frequency. Regularize time carefully; interpolation using future observations can leak information. Fit preprocessing on training history. Define forecast issue time, horizon and what observations are actually available then. Include persistence and a suitable seasonal baseline. Use multiple time blocks and site holdouts where relevant; examine errors during extremes and data gaps. A univariate LSTM does not automatically account for weather forecasts, watershed physics or delayed reporting.

## Educator note

Suggested 100 minutes: 15 for gates, 20 for windows and leakage, 15 for tensor shapes, 20 for training/baselines, and 30 for the independent experiment and peer explanation. Have students draw one input window and its target before fitting. A simpler baseline winning is an informative outcome, not a failed lesson.